# Stage 2 — Train Student (Colab)

Trains a small CNN `g: image -> R^k` to regress directly onto Stage 1's oracle compressed embeddings `e' = eW`, using **only** the same `m` fit images used to fit `W` — no further CLIP access after this point (`distillation_experiment_prompt.md`).

Loss: `mse_weight * MSE + cosine_weight * (1 - cosine_sim)` (both configurable in `configs/pilot_dog15_clip_pca.yaml`).

**Prerequisite**: `stage1_oracle_colab.ipynb` has already been run (produced `outputs/compression/<run_name>/{W,mean,fit_targets,eval_oracle}.npy` and cached CLIP embeddings). This notebook reconnects to the same Drive `outputs/` — no CLIP forward pass is needed here, only raw images + Stage 1's saved targets.

Run on a Colab GPU runtime (Runtime → Change runtime type → GPU).

In [3]:
import os

REPO_URL = "https://github.com/Nizaxga/distillation-embedding-experiment.git"
REPO_DIR = "/content/distillation-embedding-experiment"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

/content/distillation-embedding-experiment


In [ ]:
!pip install -q -r requirements.txt

## Reconnect to Stage 1's outputs

Mount the same Drive, set the same `save_dir`, and symlink `outputs/` to the same path Stage 1 wrote into — this is what makes `W.npy`/`fit_targets.npy` and the cached raw-image dir visible in this fresh runtime.

In [5]:
from google.colab import drive

drive.mount("/content/drive")

save_dir = "/content/drive/MyDrive/representation-learning"
print(f"[LOG] save_dir={save_dir}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[LOG] save_dir=/content/drive/MyDrive/representation-learning


In [6]:
os.environ["IMAGENET_DOG15_CACHE"] = os.path.join(save_dir, ".cache/imagenet-dog-15")

In [7]:
# Same DRIVE_OUTPUTS path as stage1_oracle_colab.ipynb -- must match exactly, or
# outputs/compression/<run_name>/ from Stage 1 won't be visible here.
DRIVE_OUTPUTS = os.path.join(save_dir, "output-distillation-embedding-experiment")

if not os.path.exists("outputs"):
    os.makedirs(DRIVE_OUTPUTS, exist_ok=True)
    os.symlink(DRIVE_OUTPUTS, "outputs")

In [17]:
import sys

sys.path.insert(0, REPO_DIR)

import numpy as np
import torch

from src.compress.fit import compression_dir
from src.data.imagenet_dog15 import load_dog15, split_fit_eval
from src.device import get_device
from src.students.models import build_student
from src.students.train import checkpoint_dir, train_student
from src.utils.config import load_config

In [18]:
cfg = load_config("configs/pilot_dog15_clip_pca.yaml")
device = get_device()
print(f"run_name={cfg.run_name}  arch={cfg.student_arch}  epochs={cfg.epochs}")

[device] using CUDA: Tesla T4
run_name=imagenet_dog15_clip_vit_b32_pca_k32_m1000_small_cnn_seed0  arch=small_cnn  epochs=60


## 1. Rebuild the deterministic fit/eval split

Same seed as Stage 1 -> same `fit_records` in the same order. No CLIP call needed, just re-lists the cached image dir.

In [10]:
records = load_dog15(seed=cfg.seed)
fit_records, eval_records = split_fit_eval(records, cfg.m, seed=cfg.seed)
print(f"total={len(records)}  fit={len(fit_records)}  eval={len(eval_records)}")

[data] label dirs under /content/drive/MyDrive/representation-learning/.cache/imagenet-dog-15 aren't synset ids (['0', '1', '10', '11', '12', '13', '14', '2', '3', '4', '5', '6', '7', '8', '9']); assigning class indices by sort order.
[data] loaded 1500 ImageNet-Dog-15 images from precached dir /content/drive/MyDrive/representation-learning/.cache/imagenet-dog-15
total=1500  fit=1000  eval=500


## 2. Load Stage 1's oracle targets

`fit_targets[i]` is the regression target for `fit_records[i]` -- computed in Stage 1 as `(clip_embed(image) - mean) @ W`. Loading it here (instead of recomputing) is exactly the point: no foundation-model access from this stage onward.

In [11]:
comp_dir = compression_dir(cfg.run_name)
fit_targets = np.load(comp_dir / "fit_targets.npy")
assert len(fit_targets) == len(fit_records), (
    f"{len(fit_targets)} targets vs {len(fit_records)} fit_records -- "
    "did `m` or the data dir change since Stage 1 ran?"
)
print(f"loaded fit_targets={fit_targets.shape} from {comp_dir}")

loaded fit_targets=(1000, 32) from /content/distillation-embedding-experiment/outputs/compression/imagenet_dog15_clip_vit_b32_pca_k32_m1000_small_cnn_seed0


## 3. Train the student

`train_student()` splits the fit set 90/10 train/val internally, trains `SmallCNN` with the MSE+cosine loss, prints `val_mse`/`val_cos_sim` (embedding fidelity vs oracle) every epoch, and checkpoints to `outputs/checkpoints/<run_name>/` every `checkpoint_every_steps` steps and at each epoch end -- survives a session disconnect since `outputs/` is symlinked onto Drive.

In [19]:
model = train_student(cfg, fit_records, fit_targets, device=device)

epoch 0 train: 100%|██████████| 15/15 [00:02<00:00,  5.03it/s]


[train] epoch 0: train_loss=0.8019 val_mse=0.6506 val_cos_sim=0.0987


epoch 1 train: 100%|██████████| 15/15 [00:04<00:00,  3.60it/s]


[train] epoch 1: train_loss=0.6994 val_mse=0.6513 val_cos_sim=0.1680


epoch 2 train: 100%|██████████| 15/15 [00:03<00:00,  4.72it/s]


...

[train] epoch 57: train_loss=0.2511 val_mse=0.6286 val_cos_sim=0.2761


epoch 58 train: 100%|██████████| 15/15 [00:02<00:00,  5.14it/s]


[train] epoch 58: train_loss=0.2419 val_mse=0.6331 val_cos_sim=0.2314


epoch 59 train: 100%|██████████| 15/15 [00:04<00:00,  3.21it/s]


[train] epoch 59: train_loss=0.2378 val_mse=0.6302 val_cos_sim=0.2696


## 4. Random-init sanity floor

Same architecture, zero training -- Stage 3's floor row ("student minus all training"). Seeded so it's reproducible, and saved alongside the trained checkpoints.

In [20]:
torch.manual_seed(cfg.seed)
random_model = build_student(cfg.student_arch, cfg.k)

ckpt_dir = checkpoint_dir(cfg.run_name)
torch.save(random_model.state_dict(), ckpt_dir / "random_init.pt")
print(f"random-init sanity floor saved to {ckpt_dir / 'random_init.pt'}")

random-init sanity floor saved to /content/distillation-embedding-experiment/outputs/checkpoints/imagenet_dog15_clip_vit_b32_pca_k32_m1000_small_cnn_seed0/random_init.pt


## Done

`outputs/checkpoints/<run_name>/` now has `final.pt` (trained student), per-epoch/per-step checkpoints, and `random_init.pt`. Combined with Stage 1's `outputs/compression/<run_name>/` (`W`, `mean`, `eval_oracle`), everything Stage 3 needs is in place: embed `eval_records` with the trained student and with `random_init`, cluster both plus `full`/`oracle_<method>`, and compare all four rows (see `docs/RUNNING.md` "Interpreting the output").